# PyTorch + einops Sandbox

A scratchpad for experimenting with tensor operations, broadcasting, and einops.

In [1]:
import torch
import numpy as np
from einops import rearrange, reduce, repeat

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.12.1


## 1. Create Basic Tensors

In [2]:
# 1D and 2D tensors
a = torch.arange(0, 10, 2)
b = torch.zeros(3, 4)
c = torch.ones(2, 3)
d = torch.tensor([[1., 2.], [3., 4.]])

print(f"a: shape {a.shape} = {a}")
print(f"b: shape {b.shape}")
print(f"c: shape {c.shape}")
print(f"d: shape {d.shape} (a 2D matrix)")

a: shape torch.Size([5]) = tensor([0, 2, 4, 6, 8])
b: shape torch.Size([3, 4])
c: shape torch.Size([2, 3])
d: shape torch.Size([2, 2]) (a 2D matrix)


In [3]:
# Understand squeezing and unsqueezing
matrix = torch.arange(0, 12).reshape(3, 4)
print(f"matrix: shape {matrix.shape} = \n{matrix}")
print(f"matrix unsqueezed at dim 0: shape {matrix.unsqueeze(0).shape} = \n{matrix.unsqueeze(0)}")
print(f"""Shape will be [1,3,4] here, which means, that it's a 3D tensor where the first dimension is 1, hence, you can only access the first element  
          of the tensort by using matrix[0], which will return a 2D tensor of shape [3,4] -> matrix.unsqueeze(0)[0].shape: {matrix.unsqueeze(0)[0].shape}
    """)

matrix: shape torch.Size([3, 4]) = 
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
matrix unsqueezed at dim 0: shape torch.Size([1, 3, 4]) = 
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]]])
Shape will be [1,3,4] here, which means, that it's a 3D tensor where the first dimension is 1, hence, you can only access the first element  
          of the tensort by using matrix[0], which will return a 2D tensor of shape [3,4] -> matrix.unsqueeze(0)[0].shape: torch.Size([3, 4])
    


In [4]:
# unsqueeze at dim 1
print(f"matrix unsqueezed at dim 1: shape {matrix.unsqueeze(1).shape} = \n{matrix.unsqueeze(1)}")
print(f"""Shape will be [3,1,4] here, which means, that it's a 3D tensor where the middle dimension is 1, hence, you can access 3 elements
          of the tensor by using [i] for i in range(3), each will return a 2D tensor of shape [1,4]:
          matrix.unsqueeze(1)[0].shape: {matrix.unsqueeze(1)[0].shape}
          matrix.unsqueeze(1)[1].shape: {matrix.unsqueeze(1)[1].shape}
          matrix.unsqueeze(1)[2].shape: {matrix.unsqueeze(1)[2].shape}
    """)

matrix unsqueezed at dim 1: shape torch.Size([3, 1, 4]) = 
tensor([[[ 0,  1,  2,  3]],

        [[ 4,  5,  6,  7]],

        [[ 8,  9, 10, 11]]])
Shape will be [3,1,4] here, which means, that it's a 3D tensor where the middle dimension is 1, hence, you can access 3 elements
          of the tensor by using [i] for i in range(3), each will return a 2D tensor of shape [1,4]:
          matrix.unsqueeze(1)[0].shape: torch.Size([1, 4])
          matrix.unsqueeze(1)[1].shape: torch.Size([1, 4])
          matrix.unsqueeze(1)[2].shape: torch.Size([1, 4])
    


# Now let's talk about boardcast

In [5]:
vector = torch.tensor([10, 20])
matrix = torch.tensor([[1, 2], [3, 4]])

# if we unsqueeze the vector, it will become a 2D tensor of shape [2,1], which can be broadcasted to the shape of matrix [2,2] when added to it.
unsequeezed_vector = vector.unsqueeze(1)
print(f"vector: shape {vector.shape} = {vector}")
print(f"unsequeezed_vector: shape {unsequeezed_vector.shape} = \n{unsequeezed_vector}")
print(f"matrix: shape {matrix.shape} = \n{matrix}")
print(f"matrix + unsequeezed_vector: shape {(matrix + unsequeezed_vector).shape} = \n{matrix + unsequeezed_vector}")


vector: shape torch.Size([2]) = tensor([10, 20])
unsequeezed_vector: shape torch.Size([2, 1]) = 
tensor([[10],
        [20]])
matrix: shape torch.Size([2, 2]) = 
tensor([[1, 2],
        [3, 4]])
matrix + unsequeezed_vector: shape torch.Size([2, 2]) = 
tensor([[11, 12],
        [23, 24]])


## 2. Reshape, Squeeze, and Broadcast

In [6]:
# Reshape
x = torch.arange(12)
print(f"Original: {x.shape} -> {x}")

square = x.reshape(3, 4)
print(f"Reshaped (3,4):\n{square}")

# Squeeze / Unsqueeze
t = torch.randn(1, 3, 1, 5)
print(f"\nBefore squeeze: {t.shape}")
print(f"After squeeze:  {t.squeeze().shape}")
print(f"unsqueeze(0) on (3,): {torch.randn(3).unsqueeze(0).shape}")

# Broadcasting: add vector to each column
matrix = torch.tensor([[1, 2], [3, 4]])   # (2, 2)
vec = torch.tensor([10, 20])              # (2,)
result = matrix + vec.unsqueeze(1)        # (2, 2) + (2, 1) -> broadcast
print(f"\nBroadcast add to each column:\n{matrix} + {vec} =\n{result}")

# Matrix multiplication via broadcasting
A = torch.tensor([[1., 2.], [3., 4.]])
B = torch.tensor([[5., 6.], [7., 8.]])
C = torch.sum(A.unsqueeze(2) * B.unsqueeze(0), dim=1)
print(f"\nMatmul via broadcast:\n{A} @ {B} =\n{C}")

Original: torch.Size([12]) -> tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
Reshaped (3,4):
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

Before squeeze: torch.Size([1, 3, 1, 5])
After squeeze:  torch.Size([3, 5])
unsqueeze(0) on (3,): torch.Size([1, 3])

Broadcast add to each column:
tensor([[1, 2],
        [3, 4]]) + tensor([10, 20]) =
tensor([[11, 12],
        [23, 24]])

Matmul via broadcast:
tensor([[1., 2.],
        [3., 4.]]) @ tensor([[5., 6.],
        [7., 8.]]) =
tensor([[19., 22.],
        [43., 50.]])


## 3. einops: Rearrange

In [7]:
# Flatten images: (batch, channel, h, w) -> (batch, channel*h*w)
imgs = torch.arange(2 * 3 * 4 * 4).reshape(2, 3, 4, 4)
flat = rearrange(imgs, 'b c h w -> b (c h w)')
print(f"Images: {imgs.shape} -> Flattened: {flat.shape}")

# Swap axes: (batch, time, features) -> (time, batch, features)
x = torch.randn(4, 10, 32)
swapped = rearrange(x, 'b t f -> t b f')
print(f"Swapped: {x.shape} -> {swapped.shape}")

Images: torch.Size([2, 3, 4, 4]) -> Flattened: torch.Size([2, 48])
Swapped: torch.Size([4, 10, 32]) -> torch.Size([10, 4, 32])


## 4. einops: Reduce and Repeat

In [8]:
# Reduce: mean over spatial dimensions
x = torch.arange(2 * 3 * 2 * 2, dtype=torch.float32).reshape(2, 3, 2, 2)
pooled = reduce(x, 'b c h w -> b c', 'mean')
print(f"Mean over spatial: {x.shape} -> {pooled.shape}, values:\n{pooled}")

# Repeat: single sample -> batch
single = torch.tensor([1., 2., 3.])
batched = repeat(single, 'f -> b f', b=4)
print(f"\nRepeat: {single.shape} -> {batched.shape}\n{batched}")

Mean over spatial: torch.Size([2, 3, 2, 2]) -> torch.Size([2, 3]), values:
tensor([[ 1.5000,  5.5000,  9.5000],
        [13.5000, 17.5000, 21.5000]])

Repeat: torch.Size([3]) -> torch.Size([4, 3])
tensor([[1., 2., 3.],
        [1., 2., 3.],
        [1., 2., 3.],
        [1., 2., 3.]])


In [9]:
# Understanding dim and keepdim in min()/max()

data = torch.tensor([[1., 2., 3.],
                     [5., 5., 5.]])
print(f"{'='*60}")
print(f"data: shape {data.shape}")
print(f"{data}")
print()

# --- no dim → global min/max (scalar) ---
print(f"{'─'*60}")
print("1. NO dim — global min/max (scalar)")
print(f"{'─'*60}")
print(f"   data.min()        → {data.min():.1f}     (scalar, no shape)")
print(f"   data.max()        → {data.max():.1f}     (scalar, no shape)")
print()

# --- dim=0 without keepdim → reduces rows, shape becomes (3,) ---
print(f"{'─'*60}")
print("2. dim=0, keepdim=False — reduces rows, shape (3,)")
print(f"{'─'*60}")
result = data.min(dim=0)
print(f"   data.min(dim=0)     → values: {result.values},  shape: {result.values.shape}")
result = data.max(dim=0)
print(f"   data.max(dim=0)     → values: {result.values},  shape: {result.values.shape}")
print(f"   → Collapsed column-by-column: 1st col min=1, 2nd min=2, 3rd min=3")
print()

# --- dim=1 without keepdim → reduces columns, shape becomes (2,) ---
print(f"{'─'*60}")
print("3. dim=1, keepdim=False — reduces columns, shape (2,)")
print(f"{'─'*60}")
result = data.min(dim=1)
print(f"   data.min(dim=1)     → values: {result.values},  shape: {result.values.shape}")
result = data.max(dim=1)
print(f"   data.max(dim=1)     → values: {result.values},  shape: {result.values.shape}")
print(f"   → Collapsed row-by-row: row 0 min=1, row 1 min=5")
print()

# --- dim=1 WITH keepdim → keeps dim, shape becomes (2, 1) ---
print(f"{'─'*60}")
print("4. dim=1, keepdim=True  — keeps dim as size 1, shape (2, 1)")
print(f"{'─'*60}")
result = data.min(dim=1, keepdim=True)
print(f"   data.min(dim=1, keepdim=True)  → values: shape {result.values.shape}")
print(f"                               {result.values}")
result = data.max(dim=1, keepdim=True)
print(f"   data.max(dim=1, keepdim=True)  → values: shape {result.values.shape}")
print(f"                               {result.values}")
print()

# --- BROADCASTING DEMO: why keepdim=True matters ---
print(f"{'─'*60}")
print("5. BROADCASTING DEMO — why keepdim=True is essential")
print(f"{'─'*60}")
x_min_keep   = data.min(dim=1, keepdim=True).values   # shape (2, 1)
x_min_nokeep = data.min(dim=1, keepdim=False).values  # shape (2,)

print(f"   data:              {data.shape}")
print(f"   x_min (keepdim):   {x_min_keep.shape}   ─ can broadcast against (2,3)")
print(f"   x_min (no keep):   {x_min_nokeep.shape}    ─ broadcasts as (3,2) — WRONG!")

normalized_correct = (data - x_min_keep) / (data.max(dim=1, keepdim=True).values - x_min_keep)
print(f"\n   Correct per-row MinMax (keepdim=True):")
print(f"   {normalized_correct}")

print(f"\n   WRONG (no keepdim=False) — would crash:")
print(f"   → data (2,3) - x_min_nokeep (2,) tries to broadcast", end=" ")
print(f"as (2,3) - (3,2), which fails!")
print(f"   RuntimeError: size mismatch at non-singleton dimension 1")

print(f"\n{'='*60}")
print(f"SUMMARY:")
print(f"  dim      = which axis to collapse (0=rows, 1=columns, ...)")
print(f"  keepdim  = whether to keep the collapsed axis as size 1")
print(f"  Without keepdim, shapes don't align for broadcasting!")
print(f"{'='*60}")

data: shape torch.Size([2, 3])
tensor([[1., 2., 3.],
        [5., 5., 5.]])

────────────────────────────────────────────────────────────
1. NO dim — global min/max (scalar)
────────────────────────────────────────────────────────────
   data.min()        → 1.0     (scalar, no shape)
   data.max()        → 5.0     (scalar, no shape)

────────────────────────────────────────────────────────────
2. dim=0, keepdim=False — reduces rows, shape (3,)
────────────────────────────────────────────────────────────
   data.min(dim=0)     → values: tensor([1., 2., 3.]),  shape: torch.Size([3])
   data.max(dim=0)     → values: tensor([5., 5., 5.]),  shape: torch.Size([3])
   → Collapsed column-by-column: 1st col min=1, 2nd min=2, 3rd min=3

────────────────────────────────────────────────────────────
3. dim=1, keepdim=False — reduces columns, shape (2,)
────────────────────────────────────────────────────────────
   data.min(dim=1)     → values: tensor([1., 5.]),  shape: torch.Size([2])
   data.max(

In [10]:
higher_dim_data = torch.arange(2 * 3 * 4).reshape(2, 3, 4)
print(f"\nHigher-dimensional data: shape {higher_dim_data.shape} = \n{higher_dim_data}")

global_min = higher_dim_data.min();
print(f"\nmin across all elements: {global_min} (scalar, no shape)")

dim1_min = higher_dim_data.min(dim=1)
print(f"\n if we set dim = 1, we would see a tensor of shape {dim1_min.values.shape}")
print(f"values = \n{dim1_min.values}")




Higher-dimensional data: shape torch.Size([2, 3, 4]) = 
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

min across all elements: 0 (scalar, no shape)

 if we set dim = 1, we would see a tensor of shape torch.Size([2, 4])
values = 
tensor([[ 0,  1,  2,  3],
        [12, 13, 14, 15]])


In [11]:
# Let's now talk about Min-Max normalization. This is a common technique used to scale the values of a dataset to a specific range, typically [0, 1]. The formula for Min-Max normalization is:
t = torch.tensor([[1., 2.], [3., 4.]])
min_val = t.min()
max_val = t.max()
normalized_t = (t - min_val) / (max_val - min_val)
print(f"min_val: {min_val}, max_val: {max_val}")
print(f"\nMin-Max Normalization:\nOriginal: {t}\nNormalized: {normalized_t}")

print(f"==================== Another example of Min-Max Normalization ====================")
data = torch.tensor([[1., 2., 3.],
                     [5., 5., 5.]])

x_min = data.min(dim=1, keepdim=True).values
x_max = data.max(dim=1, keepdim=True).values

print("data=", data)
print("x_min=", x_min)
print("x_max=", x_max)
print("if we do MinMax normalization, we will get:")
normalized_data = (data - x_min) / (x_max - x_min)
print(normalized_data)


min_val: 1.0, max_val: 4.0

Min-Max Normalization:
Original: tensor([[1., 2.],
        [3., 4.]])
Normalized: tensor([[0.0000, 0.3333],
        [0.6667, 1.0000]])
==================== Another example of Min-Max Normalization ====================
data= tensor([[1., 2., 3.],
        [5., 5., 5.]])
x_min= tensor([[1.],
        [5.]])
x_max= tensor([[3.],
        [5.]])
if we do MinMax normalization, we will get:
tensor([[0.0000, 0.5000, 1.0000],
        [   nan,    nan,    nan]])


## 索引与切片 — 完整示例

从 `tutorial/01_tensor_basics.py` 中学习的所有索引方式。

In [12]:
# 创建一个 3x4 的矩阵用于演示
t = torch.arange(12).reshape(3, 4)
print("t =")
print(t)
print(f"shape: {t.shape}")

t =
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
shape: torch.Size([3, 4])


In [13]:
# 1. 单个元素: t[row, col]
print("t[0, 0] =", t[0, 0].item())       # 左上角元素 → 0
print("t[2, 3] =", t[2, 3].item())       # 右下角元素 → 11
print("t[-1]   =", t[-1])                # 最后一行 → [8, 9, 10, 11]

t[0, 0] = 0
t[2, 3] = 11
t[-1]   = tensor([ 8,  9, 10, 11])


In [14]:
# 2. 行与列切片
print("t[0]      (第一行):", t[0])
print("t[:, 1]   (第二列):", t[:, 1])
print("t[:2, 1:] (前两行, 第2列起):\n", t[:2, 1:])

t[0]      (第一行): tensor([0, 1, 2, 3])
t[:, 1]   (第二列): tensor([1, 5, 9])
t[:2, 1:] (前两行, 第2列起):
 tensor([[1, 2, 3],
        [5, 6, 7]])


In [15]:
# 3. 花式索引 — 用列表取指定行
print("t[[0, 2]] (取第0行和第2行):")
print(t[[0, 2]])

t[[0, 2]] (取第0行和第2行):
tensor([[ 0,  1,  2,  3],
        [ 8,  9, 10, 11]])


In [16]:
# 4. 布尔索引 — 筛选满足条件的元素
print("t[t > 5] (所有大于5的元素):", t[t > 5])
print("t[t % 2 == 0] (所有偶数):", t[t % 2 == 0])

t[t > 5] (所有大于5的元素): tensor([ 6,  7,  8,  9, 10, 11])
t[t % 2 == 0] (所有偶数): tensor([ 0,  2,  4,  6,  8, 10])


In [17]:
# 5. 二维花式索引 — 配对提取 (row, col)
print("t[[0, 1, 2], [0, 1, 2]] (对角线元素):")
print(t[[0, 1, 2], [0, 1, 2]])

# 等价写法，更简洁
print("\nt[torch.arange(3), torch.arange(3)] (同上):")
print(t[torch.arange(3), torch.arange(3)])

print("\n💡 这就是 test_q8 取对角线的方法！")

t[[0, 1, 2], [0, 1, 2]] (对角线元素):
tensor([ 0,  5, 10])

t[torch.arange(3), torch.arange(3)] (同上):
tensor([ 0,  5, 10])

💡 这就是 test_q8 取对角线的方法！


In [18]:
# 6. 布尔索引修改 — 这正是 test_q7 的核心用法！
t2 = torch.tensor([-1, 2, -3, 4])
print("原始:", t2)

# 方法: 用布尔掩码赋值
mask = t2 < 0
print("负数位置 (t2 < 0):", mask)

t2[mask] = 0
print("替换后:", t2)

print("\n💡 一行写完: t[t < 0] = 0")

原始: tensor([-1,  2, -3,  4])
负数位置 (t2 < 0): tensor([ True, False,  True, False])
替换后: tensor([0, 2, 0, 4])

💡 一行写完: t[t < 0] = 0


## stack vs cat — 堆叠与拼接的区别

这是理解维度操作的关键。

In [19]:
# 准备两个 1D tensor
a = torch.tensor([1, 2])   # shape: (2,)
b = torch.tensor([3, 4])   # shape: (2,)

print("a =", a, "  shape:", a.shape)
print("b =", b, "  shape:", b.shape)

a = tensor([1, 2])   shape: torch.Size([2])
b = tensor([3, 4])   shape: torch.Size([2])


### cat (拼接) — 沿**已有维度**延长

想象把两排积木**首尾相连**：

In [20]:
result_cat = torch.cat([a, b])
print("cat([a, b]):")
print("  [1, 2] + [3, 4] = [1, 2, 3, 4]")
print("  shape: (2,) + (2,) →", result_cat.shape)
print("  结果:", result_cat)

print("\n👉 维度数没变（还是1D），只是变长了")

cat([a, b]):
  [1, 2] + [3, 4] = [1, 2, 3, 4]
  shape: (2,) + (2,) → torch.Size([4])
  结果: tensor([1, 2, 3, 4])

👉 维度数没变（还是1D），只是变长了


### stack (堆叠) — 创建**新维度**

想象把两本书**叠在一起**，多了一个"层"的维度：

In [21]:
result_stack = torch.stack([a, b], dim=0)
print("stack([a, b], dim=0):")
print("   [1, 2]     [[1, 2]")
print("   [3, 4]  →   [3, 4]]")
print("  shape: (2,) + (2,) →", result_stack.shape)
print("  结果:\n", result_stack)

print("\n👉 维度数变了！1D → 2D，多了一个维度")
print("   第0维大小=2（因为叠了2个），第1维大小=2（原来的长度）")

stack([a, b], dim=0):
   [1, 2]     [[1, 2]
   [3, 4]  →   [3, 4]]
  shape: (2,) + (2,) → torch.Size([2, 2])
  结果:
 tensor([[1, 2],
        [3, 4]])

👉 维度数变了！1D → 2D，多了一个维度
   第0维大小=2（因为叠了2个），第1维大小=2（原来的长度）


### 更直观的例子：2D tensor 的 stack vs cat

In [22]:
# 重新定义 a 和 b（因为前面的 cell 可能已经修改了变量）
a = torch.tensor([1, 2])
b = torch.tensor([3, 4])

print("a =", list(a))
print("b =", list(b))

# dim=0 → 新维度在最前面
s0 = torch.stack([a, b], dim=0)
print("\nstack([a, b], dim=0) → shape:", s0.shape)
print(s0)
print("解读：第0维=2个样本，每行一个")

# dim=1 → 新维度在最后面
s1 = torch.stack([a, b], dim=1)
print("\nstack([a, b], dim=1) → shape:", s1.shape)
print(s1)
print("解读：第1维=2个样本，每列一个")

print("\n👉 数据一样，但排列方式不同！")

a = [tensor(1), tensor(2)]
b = [tensor(3), tensor(4)]

stack([a, b], dim=0) → shape: torch.Size([2, 2])
tensor([[1, 2],
        [3, 4]])
解读：第0维=2个样本，每行一个

stack([a, b], dim=1) → shape: torch.Size([2, 2])
tensor([[1, 3],
        [2, 4]])
解读：第1维=2个样本，每列一个

👉 数据一样，但排列方式不同！


In [23]:
# 两张 2x3 的"图片"
img1 = torch.tensor([[1, 2, 3],
                     [4, 5, 6]])  # shape: (2, 3)

img2 = torch.tensor([[7, 8, 9],
                     [10, 11, 12]])  # shape: (2, 3)

print("img1 shape:", img1.shape)
print("img2 shape:", img2.shape)

# --- cat: 上下拼接（沿第0维） ---
cat_result = torch.cat([img1, img2], dim=0)
print("\ncat(img1, img2, dim=0) → 上下拼接:")
print(cat_result)
print("shape:", cat_result.shape, "  # 2D → 2D，只是行数变多了")

# --- stack: 叠成新维度 ---
stack_result = torch.stack([img1, img2], dim=0)
print("\nstack(img1, img2, dim=0) → 叠成第3维:")
print(stack_result)
print("shape:", stack_result.shape, "  # 2D → 3D！(2张图, 2行, 3列)")

img1 shape: torch.Size([2, 3])
img2 shape: torch.Size([2, 3])

cat(img1, img2, dim=0) → 上下拼接:
tensor([[ 1,  2,  3],
        [ 4,  5,  6],
        [ 7,  8,  9],
        [10, 11, 12]])
shape: torch.Size([4, 3])   # 2D → 2D，只是行数变多了

stack(img1, img2, dim=0) → 叠成第3维:
tensor([[[ 1,  2,  3],
         [ 4,  5,  6]],

        [[ 7,  8,  9],
         [10, 11, 12]]])
shape: torch.Size([2, 2, 3])   # 2D → 3D！(2张图, 2行, 3列)


### 2D tensor 在不同维度上 stack

更明显的效果：3种 dim 选择，shape 完全不同。

In [24]:
x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
y = torch.tensor([[7, 8, 9],
                  [10, 11, 12]])

print("原始 shape:", x.shape, "和", y.shape)
print("x =")
print(x)
print("\ny =")
print(y)

# dim=0: 新维度插在最前
s0 = torch.stack([x, y], dim=0)
print(f"\nstack dim=0 → shape: {s0.shape}")
print("新维度在最前：2张图，每张 (2,3)")

# dim=1: 新维度插在中间
s1 = torch.stack([x, y], dim=1)
print(f"\nstack dim=1 → shape: {s1.shape}")
print("新维度在中间：每行对应位置叠成 (2,) 向量")
print("例如 s1[0,0] =", list(s1[0, 0]), " ← x[0,0]和y[0,0]叠起来")

# dim=2: 新维度插在最后
s2 = torch.stack([x, y], dim=2)
print(f"\nstack dim=2 → shape: {s2.shape}")
print("新维度在最后：每个位置变成 (2,) 向量")
print("例如 s2[0,0] =", list(s2[0, 0]), " ← x[0,0]和y[0,0]叠起来")

print("\n👉 规律：新维度插在你指定的位置，原维度往后推")
print("   (2,3) + dim=0 → (2, 2, 3)")
print("   (2,3) + dim=1 → (2, 2, 3)")
print("   (2,3) + dim=2 → (2, 3, 2)")

原始 shape: torch.Size([2, 3]) 和 torch.Size([2, 3])
x =
tensor([[1, 2, 3],
        [4, 5, 6]])

y =
tensor([[ 7,  8,  9],
        [10, 11, 12]])

stack dim=0 → shape: torch.Size([2, 2, 3])
新维度在最前：2张图，每张 (2,3)

stack dim=1 → shape: torch.Size([2, 2, 3])
新维度在中间：每行对应位置叠成 (2,) 向量
例如 s1[0,0] = [tensor(1), tensor(2), tensor(3)]  ← x[0,0]和y[0,0]叠起来

stack dim=2 → shape: torch.Size([2, 3, 2])
新维度在最后：每个位置变成 (2,) 向量
例如 s2[0,0] = [tensor(1), tensor(7)]  ← x[0,0]和y[0,0]叠起来

👉 规律：新维度插在你指定的位置，原维度往后推
   (2,3) + dim=0 → (2, 2, 3)
   (2,3) + dim=1 → (2, 2, 3)
   (2,3) + dim=2 → (2, 3, 2)


### 总结

| 操作 | 维度变化 | 比喻 |
|---|---|---|
| `cat` | **不变**，一个维度变长 | 两排积木首尾相连 |
| `stack` | **+1**，多出新一维度 | 两张纸叠在一起 |

**Q10 答案：** `torch.stack([a, b], dim=0)` — 把两个 (2,) 变成 (2, 2)

## einops 入门教程

einops 提供了一种人类可读的方式来重排、合并、减少和重复张量维度。

In [27]:
from einops import rearrange, reduce, repeat

# 创建一个示例张量: (batch=2, channel=3, height=4, width=4)
imgs = torch.arange(2 * 3 * 4 * 4).float().reshape(2, 3, 4, 4)
print(f"imgs shape: {imgs.shape}")
print(f"{imgs}")
print(f"第一张图的第一个通道的第一个 2x2:\n{imgs[0, 0, :2, :2]}")

imgs shape: torch.Size([2, 3, 4, 4])
tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]],

         [[16., 17., 18., 19.],
          [20., 21., 22., 23.],
          [24., 25., 26., 27.],
          [28., 29., 30., 31.]],

         [[32., 33., 34., 35.],
          [36., 37., 38., 39.],
          [40., 41., 42., 43.],
          [44., 45., 46., 47.]]],


        [[[48., 49., 50., 51.],
          [52., 53., 54., 55.],
          [56., 57., 58., 59.],
          [60., 61., 62., 63.]],

         [[64., 65., 66., 67.],
          [68., 69., 70., 71.],
          [72., 73., 74., 75.],
          [76., 77., 78., 79.]],

         [[80., 81., 82., 83.],
          [84., 85., 86., 87.],
          [88., 89., 90., 91.],
          [92., 93., 94., 95.]]]])
第一张图的第一个通道的第一个 2x2:
tensor([[0., 1.],
        [4., 5.]])


### 1. rearrange — 重排维度

语法：`rearrange(tensor, '输入维度名 -> 输出维度名')`

用字母给每个维度起名字，箭头左边是输入，右边是输出。

In [29]:
# --- Q11: 展平图像 ---
# 把 (batch, channel, height, width) → (batch, channel*height*width)
# 括号表示合并多个维度

flat = rearrange(imgs, 'b c h w -> b (c h w)')
print(flat)
print(f"展平后 shape: {flat.shape}")  # (2, 48)
print(f"验证: flat[0] == imgs[0].flatten()? {torch.equal(flat[0], imgs[0].flatten())}")

tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
         14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26., 27.,
         28., 29., 30., 31., 32., 33., 34., 35., 36., 37., 38., 39., 40., 41.,
         42., 43., 44., 45., 46., 47.],
        [48., 49., 50., 51., 52., 53., 54., 55., 56., 57., 58., 59., 60., 61.,
         62., 63., 64., 65., 66., 67., 68., 69., 70., 71., 72., 73., 74., 75.,
         76., 77., 78., 79., 80., 81., 82., 83., 84., 85., 86., 87., 88., 89.,
         90., 91., 92., 93., 94., 95.]])
展平后 shape: torch.Size([2, 48])
验证: flat[0] == imgs[0].flatten()? True


In [ ]:
# 对比：不用 einops 怎么做？
flat_manual = imgs.reshape(imgs.shape[0], -1)
print(f"手动 reshape: {flat_manual.shape}")
print(f"结果一样? {torch.equal(flat, flat_manual)}")

print("\n👉 einops 的优势：维度名称自解释，不需要记住 -1 的含义")

In [30]:
# --- Q12: 交换维度 ---
# 把 (batch, time, features) → (time, batch, features)
# 只需要改变字母的顺序

x = torch.randn(4, 10, 32)  # (batch=4, time=10, features=32)
swapped = rearrange(x, 'b t f -> t b f')
print(f"原始 shape: {x.shape}")
print(f"交换后 shape: {swapped.shape}")  # (10, 4, 32)

# 验证数据一致性：位置 [0,0] 的数据不变
print(f"swapped[0,0] == x[0,0]? {torch.equal(swapped[0, 0], x[0, 0])}")

print("\n👉 相当于 torch.permute(x, (1, 0, 2))，但更易读")

原始 shape: torch.Size([4, 10, 32])
交换后 shape: torch.Size([10, 4, 32])
swapped[0,0] == x[0,0]? True

👉 相当于 torch.permute(x, (1, 0, 2))，但更易读


### 2. reduce — 维度归约

语法：`reduce(tensor, '输入维度 -> 输出维度', '操作')`

操作可以是：'mean', 'sum', 'max', 'min' 等

In [32]:
# --- Q13: 对空间维度 (h, w) 求均值 ---
# 把 (batch, channel, height, width) → (batch, channel)
# 相当于全局平均池化

x = torch.arange(2 * 3 * 2 * 2, dtype=torch.float32).reshape(2, 3, 2, 2)
print(f"原始 shape: {x.shape}")

pooled = reduce(x, 'b c h w -> b c', 'mean')
print(f"reduce 后 shape: {pooled.shape}")  # (2, 3)
print(f"pooled 值:\n{pooled}")

# 对比：不用 einops
pooled_manual = x.mean(dim=(2, 3))
print(f"手动 mean(dim=(2,3)) shape: {pooled_manual.shape}")
print(f"结果一样? {torch.allclose(pooled, pooled_manual)}")

print("\npooled 值:")
print(pooled)
print("\n👉 每个值是对应 channel 上所有 h×w 位置的平均")

原始 shape: torch.Size([2, 3, 2, 2])
reduce 后 shape: torch.Size([2, 3])
pooled 值:
tensor([[ 1.5000,  5.5000,  9.5000],
        [13.5000, 17.5000, 21.5000]])
手动 mean(dim=(2,3)) shape: torch.Size([2, 3])
结果一样? True

pooled 值:
tensor([[ 1.5000,  5.5000,  9.5000],
        [13.5000, 17.5000, 21.5000]])

👉 每个值是对应 channel 上所有 h×w 位置的平均


In [33]:
# 看一个具体例子：单个 channel 的均值计算
single = torch.tensor([[1., 2.],
                        [3., 4.]])
print(f"单个 channel:\n{single}")
print(f"均值: {single.mean():.2f}")  # (1+2+3+4)/4 = 2.5

# 用 einops
single_4d = single.unsqueeze(0).unsqueeze(0)  # (1, 1, 2, 2)
result = reduce(single_4d, 'b c h w -> b c', 'mean')
print(f"einops reduce 结果: {result.item():.2f}")

单个 channel:
tensor([[1., 2.],
        [3., 4.]])
均值: 2.50
einops reduce 结果: 2.50


### 3. repeat — 重复扩展

语法：`repeat(tensor, '输入维度 -> 输出维度', 新维度大小)`

In [34]:
# --- Q14: 将单个样本扩展成 batch ---
# 把 (features,) → (batch_size, features)

single = torch.tensor([1, 2, 3])
print(f"原始: {single}, shape: {single.shape}")

batched = repeat(single, 'f -> b f', b=2)
print(f"repeat 后:\n{batched}, shape: {batched.shape}")

print("\n👉 相当于 torch.stack([single, single]) 或 single.unsqueeze(0).expand(2, -1)")

原始: tensor([1, 2, 3]), shape: torch.Size([3])
repeat 后:
tensor([[1, 2, 3],
        [1, 2, 3]]), shape: torch.Size([2, 3])

👉 相当于 torch.stack([single, single]) 或 single.unsqueeze(0).expand(2, -1)


In [ ]:
# 对比不同 repeat 方式
single = torch.tensor([1, 2, 3])

# einops
r1 = repeat(single, 'f -> b f', b=3)
print("repeat('f -> b f', b=3):")
print(r1)

# 把 1D 重复成 2D 网格
r2 = repeat(single, 'f -> f repeat', repeat=2)
print("\nrepeat('f -> f repeat', repeat=2):")
print(r2)

# 图像重复：(C, H, W) → (B, C, H, W)
img = torch.ones(3, 4, 4)  # 单张 RGB 图
img_batch = repeat(img, 'c h w -> b c h w', b=5)
print(f"\n单张图 {img.shape} → batch {img_batch.shape}")

### einops 速查表

| 函数 | 用途 | 示例 |
|------|------|------|
| `rearrange` | 重排/合并/拆分维度 | `'b c h w -> b (c h w)'` |
| `reduce` | 按维度聚合 | `'b c h w -> b c', 'mean'` |
| `repeat` | 复制扩展维度 | `'f -> b f', b=4` |

**记忆要点：**
- 箭头左边 = 输入维度名，右边 = 输出维度名
- 括号 `()` = 合并多个维度
- 新出现的字母 = 新维度（需要指定大小）
- 消失的字母 = 被 reduce 掉（需要指定操作）